# 实践2：神经网络的梯度

网络结构：**X → Linear → ReLU → Linear → Sigmoid → 二元交叉熵**

数据取自 Task1 的 make_moons，先只用 **1 个样本**，隐藏层故意设成 **3 个神经元**（方便手算和全部打印）。参数验证成功后再换成 8 个样本的小 batch。

## 形状总表（单个样本）

| 量 | 形状 | 说明 |
|---|---|---|
| X | (1, 2) | 一个样本的两个特征 |
| W1 | (2, 3) | 第一层权重 |
| b1 | (3,) | 第一层偏置 |
| Z1 = X@W1 + b1 | (1, 3) | 第一层线性输出 |
| A1 = ReLU(Z1) | (1, 3) | 激活后 |
| W2 | (3, 1) | 第二层权重 |
| b2 | (1,) | 第二层偏置 |
| Z2 = A1@W2 + b2 | (1, 1) | logit |
| P = σ(Z2) | (1, 1) | 预测是类别 1 的概率 |
| L = BCE(P, y) | 标量 | 二元交叉熵损失 |

## 计算图

```
X(1,2) ──► Z1 = X@W1+b1 ──► A1 = ReLU(Z1) ──► Z2 = A1@W2+b2 ──► P = σ(Z2) ──► L = BCE(P, y)
        └─ W1(2,3) b1(3,)                └─ W2(3,1) b2(1,)
```

## 反向公式（链式法则，从后往前）

最关键的一步：**Sigmoid 和二元交叉熵组合后，loss 对 logit 的梯度就是 P − y**。

- dL/dZ2 = P − y，形状 (1, 1)
- dW2 = A1ᵀ @ dZ2，形状 (3, 1)
- db2 = dZ2 按列求和，形状 (1,)
- dA1 = dZ2 @ W2ᵀ，形状 (1, 3)
- dZ1 = dA1 ⊙ (Z1 > 0)（ReLU 的局部导数是 0 或 1），形状 (1, 3)
- dW1 = Xᵀ @ dZ1，形状 (2, 3)
- db1 = dZ1 按列求和，形状 (3,)

下面先用 NumPy 完整走一遍前向和反向，再用 PyTorch 相同输入、相同参数验证。


In [23]:
import numpy as np
from sklearn.datasets import make_moons

X_all, y_all = make_moons(n_samples=100, noise=0.1, random_state=42)
X = X_all[0:1]                              # (1, 2) 第 0 个样本
y = np.array([[y_all[0]]], dtype=float)     # (1, 1) 它的标签
print("X =", X, " shape:", X.shape)
print("y =", y, " shape:", y.shape)


X = [[ 1.58202308 -0.44581483]]  shape: (1, 2)
y = [[1.]]  shape: (1, 1)


In [24]:
rng = np.random.default_rng(0)
W1 = rng.normal(0, 0.5, size=(2, 3))    # (2, 3)
b1 = np.zeros(3)                        # (3,)
W2 = rng.normal(0, 0.5, size=(3, 1))    # (3, 1)
b2 = np.zeros(1)                        # (1,)
print("W1 shape:", W1.shape, "\nW1 =", np.round(W1, 4))
print("b1 shape:", b1.shape, " b1 =", b1)
print("W2 shape:", W2.shape, "\nW2 =", np.round(W2, 4))
print("b2 shape:", b2.shape, " b2 =", b2)


W1 shape: (2, 3) 
W1 = [[ 0.0629 -0.0661  0.3202]
 [ 0.0525 -0.2678  0.1808]]
b1 shape: (3,)  b1 = [0. 0. 0.]
W2 shape: (3, 1) 
W2 = [[ 0.652 ]
 [ 0.4735]
 [-0.3519]]
b2 shape: (1,)  b2 = [0.]


In [25]:
Z1 = X @ W1 + b1             # (1,2)@(2,3) + (3,) -> (1,3)
A1 = np.maximum(0, Z1)       # (1,3) ReLU
Z2 = A1 @ W2 + b2            # (1,3)@(3,1) + (1,) -> (1,1)
P = 1 / (1 + np.exp(-Z2))    # (1,1) sigmoid
L = -(y * np.log(P) + (1 - y) * np.log(1 - P)).item()   # 标量
print("Z1 =", np.round(Z1, 4), " shape:", Z1.shape)
print("A1 =", np.round(A1, 4), " shape:", A1.shape)
print("Z2 =", np.round(Z2, 4), " shape:", Z2.shape)
print("P  =", np.round(P, 4), " shape:", P.shape)
print("L  =", round(L, 6))


Z1 = [[0.0761 0.0149 0.426 ]]  shape: (1, 3)
A1 = [[0.0761 0.0149 0.426 ]]  shape: (1, 3)
Z2 = [[-0.0932]]  shape: (1, 1)
P  = [[0.4767]]  shape: (1, 1)
L  = 0.740848


In [26]:
dZ2 = P - y                  # (1,1) sigmoid+BCE 融合后的梯度
dW2 = A1.T @ dZ2             # (3,1)@(1,1) -> (3,1)
db2 = dZ2.sum(axis=0)        # (1,)
dA1 = dZ2 @ W2.T             # (1,1)@(1,3) -> (1,3)
dZ1 = dA1 * (Z1 > 0)         # (1,3) ReLU 局部导数 0/1
dW1 = X.T @ dZ1              # (2,1)@(1,3) -> (2,3)
db1 = dZ1.sum(axis=0)        # (3,)
for name, g in [("dW1", dW1), ("db1", db1), ("dW2", dW2), ("db2", db2)]:
    print(name, " shape:", g.shape, " value:", np.round(g, 6))


dW1  shape: (2, 3)  value: [[-0.539763 -0.392024  0.291296]
 [ 0.152106  0.110473 -0.082087]]
db1  shape: (3,)  value: [-0.341186 -0.247799  0.184129]
dW2  shape: (3, 1)  value: [[-0.039807]
 [-0.007801]
 [-0.222911]]
db2  shape: (1,)  value: [-0.523291]


In [27]:
import torch
import torch.nn.functional as F

# 参数从 NumPy 原值复制，保证"相同输入和参数"；用 float64 对齐精度方便比较
tX = torch.tensor(X, dtype=torch.float64)
ty = torch.tensor(y, dtype=torch.float64)
tW1 = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
tb1 = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
tW2 = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
tb2 = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

tZ1 = tX @ tW1 + tb1
tA1 = torch.relu(tZ1)
tZ2 = tA1 @ tW2 + tb2
loss = F.binary_cross_entropy_with_logits(tZ2, ty)   # 数学上等价于 sigmoid + BCE
loss.backward()
print("loss =", loss.item())


loss = 0.7408484906952997


In [28]:
pairs = [("dW1", dW1, tW1.grad), ("db1", db1, tb1.grad),
         ("dW2", dW2, tW2.grad), ("db2", db2, tb2.grad)]
for name, g_np, g_t in pairs:
    g_t = g_t.numpy()
    err = np.abs(g_np - g_t).max()
    print(f"{name}: 形状 NumPy {g_np.shape} vs PyTorch {g_t.shape} | "
          f"最大绝对误差 {err:.2e} | 一致: {err < 1e-10}")


dW1: 形状 NumPy (2, 3) vs PyTorch (2, 3) | 最大绝对误差 0.00e+00 | 一致: True
db1: 形状 NumPy (3,) vs PyTorch (3,) | 最大绝对误差 0.00e+00 | 一致: True
dW2: 形状 NumPy (3, 1) vs PyTorch (3, 1) | 最大绝对误差 1.73e-18 | 一致: True
db2: 形状 NumPy (1,) vs PyTorch (1,) | 最大绝对误差 0.00e+00 | 一致: True


## 换成小 batch（8 个样本）

batch 之后只有三处变化：

1. 所有中间结果的**样本维**从 1 变成 8（X 是 (8, 2)，Z1 是 (8, 3)……）；
2. loss 从单个样本的 BCE 变成 **8 个样本的平均值**；
3. 反向的开头 dL/dZ2 从 P − y 变成 **(P − y) / 8**（多出来的平均要在梯度里除回来）。

后面所有公式**形式不变**——矩阵乘法自动处理样本维。下面验证。


In [29]:
Xb = X_all[0:8]                                # (8, 2)
yb = y_all[0:8].reshape(-1, 1).astype(float)   # (8, 1)
n = Xb.shape[0]

Z1b = Xb @ W1 + b1                    # (8,2)@(2,3) -> (8,3)
A1b = np.maximum(0, Z1b)              # (8,3)
Z2b = A1b @ W2 + b2                   # (8,3)@(3,1) -> (8,1)
Pb = 1 / (1 + np.exp(-Z2b))           # (8,1)
Lb = np.mean(-(yb * np.log(Pb) + (1 - yb) * np.log(1 - Pb)))   # 平均损失

dZ2b = (Pb - yb) / n                  # (8,1) 平均损失让梯度多除一个 n
dW2b = A1b.T @ dZ2b                   # (3,8)@(8,1) -> (3,1)
db2b = dZ2b.sum(axis=0)               # (1,)
dA1b = dZ2b @ W2.T                    # (8,1)@(1,3) -> (8,3)
dZ1b = dA1b * (Z1b > 0)               # (8,3)
dW1b = Xb.T @ dZ1b                    # (2,8)@(8,3) -> (2,3)
db1b = dZ1b.sum(axis=0)               # (3,)
print("batch loss =", round(Lb, 6))
for name, g in [("dW1", dW1b), ("db1", db1b), ("dW2", dW2b), ("db2", db2b)]:
    print(name, " shape:", g.shape, " value:", np.round(g, 6))


batch loss = 0.703396
dW1  shape: (2, 3)  value: [[-0.150378 -0.144524  0.081155]
 [ 0.068896  0.046574 -0.037181]]
db1  shape: (3,)  value: [-0.127042 -0.061804  0.068561]
dW2  shape: (3, 1)  value: [[-0.008957]
 [-0.006183]
 [-0.054749]]
db2  shape: (1,)  value: [-0.007278]


In [30]:
tXb = torch.tensor(Xb, dtype=torch.float64)
tyb = torch.tensor(yb, dtype=torch.float64)
for p in [tW1, tb1, tW2, tb2]:
    p.grad = None                     # 清掉单样本那次的梯度

tZ1b = tXb @ tW1 + tb1
tA1b = torch.relu(tZ1b)
tZ2b = tA1b @ tW2 + tb2
lossb = F.binary_cross_entropy_with_logits(tZ2b, tyb)   # 默认就是对 batch 求平均
lossb.backward()

pairs_b = [("dW1", dW1b, tW1.grad), ("db1", db1b, tb1.grad),
           ("dW2", dW2b, tW2.grad), ("db2", db2b, tb2.grad)]
for name, g_np, g_t in pairs_b:
    g_t = g_t.numpy()
    err = np.abs(g_np - g_t).max()
    print(f"{name}: 形状 NumPy {g_np.shape} vs PyTorch {g_t.shape} | "
          f"最大绝对误差 {err:.2e} | 一致: {err < 1e-10}")


dW1: 形状 NumPy (2, 3) vs PyTorch (2, 3) | 最大绝对误差 0.00e+00 | 一致: True
db1: 形状 NumPy (3,) vs PyTorch (3,) | 最大绝对误差 0.00e+00 | 一致: True
dW2: 形状 NumPy (3, 1) vs PyTorch (3, 1) | 最大绝对误差 8.67e-19 | 一致: True
db2: 形状 NumPy (1,) vs PyTorch (1,) | 最大绝对误差 0.00e+00 | 一致: True


## 单样本 → 小 batch，变与不变

**变的：**

- 形状：所有中间结果和 dZ1、dA1、dZ2 的样本维从 1 变 8；参数梯度 dW1、db1、dW2、db2 形状不变（参数本来就和样本数无关）；
- loss 变成 8 个样本的平均；
- 反向开头多一步除以 n：(P − y) / 8。

**不变的：**

- dW2 = A1ᵀ @ dZ2、dW1 = Xᵀ @ dZ1 这些公式一行没改，矩阵乘法自动把 8 个样本的贡献汇总；
- Sigmoid + BCE 融合成 P − y 这个结论依然成立。

这就是深度学习框架能同时处理任意 batch size 的原因：反向公式按样本维写成矩阵运算，batch 大小只是矩阵的形状参数。
